# Training on SHD: do the networks use the order of events, and what does it cost?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Avi-Seth/Energy-Accuracy-Tradeoff-Benchmarking-on-Spatio-Temporal-Datasets/blob/main/02_train_SHD.ipynb)

**The question.** A spoken word in SHD (Spiking Heidelberg Digits) is a list of spikes on 700 channels.
We cut every recording into time windows and count the spikes of every channel in every window. This
gives a table with one row per window. Some networks read the rows **in order** (spiking networks, a
recurrent network, LSTMs). Do they really use that order, or would a network that never sees the order
do just as well?

Three kinds of model read the same tables:

- the **count classifier** sees only each channel's total number of spikes over the whole recording;
- **order-blind designs** look at every window but cannot see the order of the windows (they add up,
  average or take the largest value over the windows). The best of them, chosen on held-out
  recordings, is the **order-blind ceiling r**;
- **order-using networks** read the windows in order. Each one is trained twice from the same seed:
  with the windows in order (test accuracy **a0**) and with every recording's windows shuffled (**a1**).

```
drop = a0 - a1 = (a0 - r) + (r - a1) = gain + deficit
```

The order-blind side has nine designs: the eight used on SSC in `03_train_SSC.ipynb` (the count
classifier, summed scores per window, two mean-pooled designs, and a deeper mean-and-maximum design
at four levels of input dropout), and one design at the size of RadLIF, the largest network here.

A network can only be said to need the order by at most its **gain** over the ceiling. This notebook
trains the models and saves their results, including an energy estimate for every model (computed
from counts of operations, not measured on a chip). The notebook `04_analyze_results.ipynb` then
computes the gains and makes the tables and figures.

**What this notebook runs** (SHD only; `03_train_SSC.ipynb` runs SSC. The two do not depend on each
other, so they can run at the same time in two Colab sessions if your Colab account allows two GPU
sessions at once):

| part | what | runs | hours on an NVIDIA L4 GPU |
|---|---|---|---|
| 1 | control tasks, whose answers are known in advance | 150 | about 1.6 |
| 2 | every recording cut into 25 windows of equal length | 125 | under 2 |
| 3 | every recording cut into 100 windows of 14 ms | 105 | under 7 |

The hours are planning figures. Before training, every part times each model on the GPU you have and
prints its own estimate; a progress bar then shows the time left.

**How to run it.** In Colab choose a GPU (Runtime > Change runtime type), then Runtime > Run all.
Nothing else is needed: the SHD files are downloaded from the dataset authors' site. As it is set, the
notebook runs a **quick test** that takes a few minutes and uses a few recordings, so its numbers mean
nothing. For the full study, set `QUICK = False` in Step 1. Colab then asks once for access to your
Google Drive, and every finished run is saved there at once. A full run can last longer than a Colab
session; if the session stops, open the notebook again and run all the cells: runs already saved are
skipped.

## Step 1. Settings

- `QUICK = True` tests the whole notebook in a few minutes, with a few recordings and one or two
  training epochs; its numbers mean nothing. `QUICK = False` runs the full study.
- `USE_DRIVE = True` keeps the results on your Google Drive. A full run always does, because a Colab
  machine and everything on it is deleted when the session ends.
- `MY_DATA_DIR` is optional: a folder that already holds `shd_train.h5` and `shd_test.h5` (or their
  `.h5.gz` versions), anywhere inside it. When it is empty, the files are downloaded.
- `RUN_PART_1` to `RUN_PART_3` choose the parts to run.

In [ ]:
QUICK = True              # True: a quick test (a few minutes; its numbers mean nothing). False: the full study
USE_DRIVE = False         # True: keep the results on your Google Drive (a full run always does)
MY_DATA_DIR = ''          # optional: a folder with the SHD files, for example '/content/drive/MyDrive/data/'

RUN_PART_1 = True         # part 1: the control tasks
RUN_PART_2 = True         # part 2: 25 windows per recording
RUN_PART_3 = True         # part 3: 100 windows of 14 ms
RUN_CHECKS = True         # the self-checks before training (they take seconds)

## Step 2. Folders, Google Drive and snnTorch

The results of a full run go to Google Drive, so that they outlast the Colab session; Colab asks once
for permission. A quick test keeps everything on the Colab machine, and its last cell offers the
results as a download. snnTorch is a library of spiking neurons. It is used only by the self-checks
(step 11), to confirm that the spiking neurons written out in this notebook behave exactly like
snnTorch's.

In [ ]:
if not QUICK and not USE_DRIVE:               # a full run needs a place that outlasts the session
    USE_DRIVE = True
    print('A full run keeps its results on Google Drive, so USE_DRIVE is set to True.')
if USE_DRIVE or MY_DATA_DIR.startswith('/content/drive'):   # Google Drive is needed: connect it
    from google.colab import drive            # the tool that connects Google Drive to this notebook
    drive.mount('/content/drive')             # your Drive files appear under /content/drive/MyDrive

if USE_DRIVE:
    BASE_DIR = '/content/drive/MyDrive/shd_ssc_order/'    # on your Drive: kept after the session
else:
    BASE_DIR = '/content/shd_ssc_order/'                  # on this Colab machine: deleted after the session
if QUICK:
    RESULTS_DIR = BASE_DIR + 'results_quick/'             # a quick test writes to its own folder
else:
    RESULTS_DIR = BASE_DIR + 'results/'                   # where every finished run is saved
DATA_DIR = '/content/data/'                               # the data files, on this machine's own disk (fast)
DATA_URL = 'https://zenkelab.org/datasets/'               # the dataset authors' site
print('results folder:', RESULTS_DIR)
%pip install -q snntorch==1.0.0

## Step 3. Libraries

The tools this notebook uses. The last lines check that a GPU is available.

In [ ]:
import os                                     # folders and file names
import json                                   # saving results as text files
import time                                   # measuring how long things take
import shutil                                 # copying files and making .zip files
import gzip                                   # unpacking the downloaded .gz files
import hashlib                                # checksums of the downloaded files
import urllib.request                         # downloading files
from itertools import permutations            # the six orders of three words (part 1)

import h5py                                   # reading the .h5 data files
import numpy as np                            # arrays of numbers
import torch                                  # the neural-network library
import torch.nn as nn                         # ready-made network layers
import torch.nn.functional as F               # functions such as softmax
from tqdm.auto import tqdm                    # progress bars

if torch.cuda.is_available():                 # is there a GPU?
    DEVICE = 'cuda'                           # yes: compute on it
    print('GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = 'cpu'                            # no: training would be very slow
    print('No GPU. In Colab choose Runtime > Change runtime type > a GPU, then run the notebook again.')

## Step 4. The fixed numbers of the study

These numbers define the study. Every model is trained with five different seeds (a seed fixes the
random starting weights), so every result comes with a spread. A recipe says how a model is trained:
the optimiser, its learning rate, how many epochs, and whether the last epoch is kept or the epoch
that did best on the held-out recordings.

In [ ]:
C = 700                        # input channels (SHD's 700 ear-model channels)
K = 20                         # classes: the digits 0 to 9 spoken in English and in German
H = 128                        # hidden units of the small models
BATCH = 128                    # recordings per training step
SEEDS = [13, 29, 43, 61, 73]   # five seeds: every model is trained five times
HOLDOUT = 0.10                 # 10% of the training recordings are held out to choose designs
VAL_SEED = 2027                # seed of the random choice of held-out recordings
EVAL_SEED = 17                 # EVAL_SEED + seed fixes RadLIF's random starting states when it is tested
SHUFFLE_SEED_FIT = 1000        # + 10 x seed: window orders of the shuffled fitting recordings
SHUFFLE_SEED_VAL = 1500        # + 10 x seed: ... of the shuffled held-out recordings
SHUFFLE_SEED_TEST = 2000       # + 10 x seed: ... of the shuffled test recordings
EVAL_ONLY_SEED = 3000          # + 10 x seed + i: shuffling at test time only (part 2)
NEGATIVE_SEED_FIT = 4000       # + 10 x seed: the negative control's first shuffle of its training samples
NEGATIVE_SEED_TEST = 5000      # + 10 x seed: ... of its test samples
BETA_25 = 0.9                  # spiking neurons keep 90% of their voltage per window at 25 windows (about 28 ms)
BETA_14 = 0.9 ** (14 / 28)     # the same memory in milliseconds for 14 ms windows (about 0.949 per window)

RECIPE_SHD = {'optimizer': 'adam', 'lr': 0.002, 'epochs': 40, 'keep': 'final'}           # most models
RECIPE_RADLIF = {'optimizer': 'adam', 'lr': 0.01, 'epochs': 60, 'keep': 'best', 'plateau': True}   # RadLIF's

# the 9 order-blind designs: the 8 of the SSC notebook and one at RadLIF's size
# (the order of this list only matters to break an exact tie)
DESIGNS = ['count_256', 'perminv', 'pool_128', 'pool_512', 'meanmax_deep_128', 'meanmax_deep_128_d20',
           'meanmax_deep_128_d50', 'meanmax_deep_128_d70', 'ob_deep_1024_d10']
LARGE_DESIGNS = ['ob_deep_1024_d10']                                              # the design at RadLIF's scale
NETWORKS_25 = ['lif_std', 'rlif', 'rnn', 'lstm_d0', 'lstm_d20', 'lstm_d50', 'lstm_d70', 'radlif']    # part 2
NETWORKS_14 = ['radlif', 'rlif', 'lstm_d0', 'lstm_d20', 'lstm_d50', 'lstm_d70']                      # part 3
CONTROL_DESIGNS = ['count_256', 'pool_128']                                                          # part 1
CONTROL_NETWORKS = ['lif_std', 'rlif', 'rnn', 'lstm']                                                # part 1

if QUICK:
    SEEDS_USED = SEEDS[:2]                    # a quick test uses two seeds
else:
    SEEDS_USED = SEEDS


def recipe_of(name):
    """The training recipe of a model (in a quick test: 1 epoch, or 2 when the best epoch is kept)."""
    if name in ('radlif', 'ob_deep_1024_d10'):    # RadLIF, and the order-blind design of its size
        recipe = dict(RECIPE_RADLIF)              # dict(...) makes a copy we may change
    else:
        recipe = dict(RECIPE_SHD)                 # every other model
    if QUICK:                                     # a quick test: as few epochs as possible
        if recipe['keep'] == 'best':
            recipe['epochs'] = 2
        else:
            recipe['epochs'] = 1
    return recipe

## Step 5. Get the SHD files

SHD has two files, `shd_train.h5` (8,156 recordings) and `shd_test.h5` (2,264 recordings); downloaded,
they are 169 MB together.

For every file the notebook looks, in this order:

1. on this machine, from earlier in the same session;
2. in `MY_DATA_DIR`, if it is set (the file or its `.gz` version, anywhere inside that folder);
3. on the dataset authors' site, https://zenkelab.org/datasets/. The compressed file is downloaded,
   checked against the checksum the site publishes (an MD5 checksum is a fingerprint of a file, so a
   damaged download is caught), and unpacked.

The files go to the Colab machine's own disk, which is fast to read. Every file is written under a
temporary name and renamed only when complete, so an interrupted copy or download is never used.

In [ ]:
def find_in_my_data(name):
    """The path of `name` (or of `name` + '.gz') anywhere inside MY_DATA_DIR, or None if it is not there."""
    if MY_DATA_DIR == '':                                              # no folder of your own was given
        return None
    for folder, subfolders, files in os.walk(MY_DATA_DIR):            # every folder inside MY_DATA_DIR
        subfolders.sort()                                              # look through them in alphabetical order
        for candidate in [name, name + '.gz']:                         # the file itself, or its compressed version
            if candidate in files:
                return os.path.join(folder, candidate)
    return None


def download(url, path):
    """Download `url` to `path` with a progress bar; the file gets its real name only when complete."""
    with urllib.request.urlopen(url, timeout=60) as response:          # open the connection
        size = int(response.headers.get('Content-Length', 0))          # the file's size, if the site gives it
        with open(path + '.part', 'wb') as out:
            with tqdm(total=size, unit='B', unit_scale=True, desc='downloading ' + os.path.basename(path)) as bar:
                while True:
                    chunk = response.read(4 * 1024 * 1024)             # 4 MB at a time
                    if len(chunk) == 0:                                # nothing left: the download is complete
                        break
                    out.write(chunk)
                    bar.update(len(chunk))
    os.replace(path + '.part', path)                                   # the real name, now that it is complete


def md5_of(path):
    """The MD5 checksum of a file: a fingerprint that changes if a single byte of the file changes."""
    digest = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(16 * 1024 * 1024)                           # 16 MB at a time
            if len(chunk) == 0:
                break
            digest.update(chunk)
    return digest.hexdigest()


def published_md5(dataset):
    """The checksums the dataset authors publish for 'shd' or 'ssc', as {file name: checksum}."""
    with urllib.request.urlopen(DATA_URL + dataset + '/md5sums.txt', timeout=60) as response:
        text = response.read().decode()
    sums = {}
    for line in text.splitlines():                                     # every line is 'checksum  file name'
        parts = line.split()
        if len(parts) == 2:
            sums[parts[1]] = parts[0]
    return sums


def unpack(packed, path):
    """Unpack a .gz file to `path`; the file gets its real name only when complete."""
    with gzip.open(packed, 'rb') as source:
        with open(path + '.part', 'wb') as out:
            shutil.copyfileobj(source, out, 16 * 1024 * 1024)          # 16 MB at a time
    os.replace(path + '.part', path)


def get_data_file(name):
    """Put the data file `name` (for example 'shd_train.h5') on this machine, and return where it is."""
    path = DATA_DIR + name
    if os.path.exists(path):                                           # 1. here from earlier in this session
        return path
    mine = find_in_my_data(name)
    if mine is not None:                                               # 2. in MY_DATA_DIR
        print('taking', name, 'from', mine)
        if mine.endswith('.gz'):
            unpack(mine, path)                                         # a compressed file: unpack it here
        else:
            shutil.copy(mine, path + '.part')                          # copy it here, under a temporary name
            os.replace(path + '.part', path)
        return path
    dataset = name.split('_')[0]                                       # 3. from the dataset authors' site
    packed = path + '.gz'                                              # the site has compressed files only
    url = DATA_URL + dataset + '/' + name + '.gz'
    try:
        download(url, packed)
    except Exception as error:                                         # no connection, a moved file, ...
        raise RuntimeError(f'{name}.gz could not be downloaded from {url} ({error}). Download the '
                           f'{dataset.upper()} files from https://zenkelab.org/datasets/ yourself, put them in a '
                           f'folder on your Google Drive, set MY_DATA_DIR in Step 1 to that folder, and run '
                           f'the notebook again.') from None
    try:
        expected = published_md5(dataset).get(name + '.gz')            # the checksum the site publishes
    except Exception:
        expected = None                                                # the list of checksums could not be read
    if expected is None:
        print('note: no published checksum was found for', name + '.gz', '- the download could not be checked')
    elif md5_of(packed) != expected:                                   # a damaged download
        os.remove(packed)
        raise RuntimeError(f'{name}.gz was damaged while downloading (its checksum is wrong). '
                           f'Run this cell again.')
    else:
        print(name + '.gz: its checksum matches the one the dataset authors publish')
    print('unpacking', name + '.gz')
    unpack(packed, path)
    os.remove(packed)                                                  # the compressed file is no longer needed
    return path


os.makedirs(DATA_DIR, exist_ok=True)                                   # create the data folder if needed
SHD_TRAIN = get_data_file('shd_train.h5')                              # the training recordings
SHD_TEST = get_data_file('shd_test.h5')                                # the test recordings
print('SHD files on this machine:', SHD_TRAIN, SHD_TEST)

## Step 6. From spikes to a table of counts

Every recording is a list of spike times (in seconds) and channels (0 to 699). We cut it into T time
windows and count, for every window and every channel, how many spikes fell there. Two ways to cut:

- **relative**: T windows spanning the recording from its first spike to its last, so every window
  lasts 1/T of that recording (25 windows of about 28 ms each, on average);
- **fixed**: T windows of a fixed width from time 0 (100 windows of 14 ms); a short recording ends with
  empty windows, and a spike after the last window is counted in the last window.

The files store spike times in half precision (16-bit numbers). They are converted to double
precision before any arithmetic, because in half precision some spikes would land in a neighbouring
window.

In [ ]:
def count_spikes(path, how, T, width_s=0.0):
    """Count the spikes of every recording of one SHD file per (window, channel).
    Returns the table (recordings x T x 700), the classes, the speakers and every recording's length."""
    with h5py.File(path, 'r') as f:                               # open the file for reading
        times = f['spikes']['times']                              # every recording's spike times (seconds)
        units = f['spikes']['units']                              # every spike's channel (0 to 699)
        labels = np.array(f['labels']).astype(np.int64)           # every recording's class (0 to 19)
        speakers = np.array(f['extra']['speaker']).astype(np.int64)   # who spoke each recording
        n = len(labels)                                           # how many recordings
        X = np.zeros((n, T, C), dtype=np.uint16)                  # the table, filled below
        durations = np.zeros(n)                                   # each recording's length, first to last spike
        for start in tqdm(range(0, n, 1024), desc='counting spikes in ' + os.path.basename(path)):
            block_times = times[start:start + 1024]               # 1,024 recordings at a time: reading many at
            block_units = units[start:start + 1024]               # once is much faster than one by one
            for k in range(len(block_times)):                     # one recording at a time
                t = block_times[k].astype(np.float64)             # its spike times, in double precision
                ch = block_units[k].astype(np.int64)              # its spike channels
                if len(t) > 0:
                    durations[start + k] = t.max() - t.min()      # its length in seconds
                if how == 'relative':                             # T windows spanning this recording
                    if len(t) > 1 and t.max() > t.min():
                        window = ((t - t.min()) / (t.max() - t.min()) * T).astype(np.int64)   # each spike's window
                        window = window.clip(0, T - 1)            # the very last spike belongs to the last window
                    else:
                        window = np.zeros(len(t), np.int64)       # no length at all: every spike in window 0
                else:                                             # fixed windows of width_s seconds from time 0
                    window = (t / width_s).astype(np.int64)       # window number = time / width
                    window = window.clip(0, T - 1)                # spikes after the last window are counted in it
                cell = window * C + ch                            # one number per (window, channel) cell
                counts = np.bincount(cell, minlength=T * C)       # how many spikes fell in every cell
                X[start + k] = counts.reshape(T, C)               # stored as a T x 700 table
    return X, labels, speakers, durations

## Step 7. Helper functions for the data

- `holdout` sets aside 10% of every class of the training recordings. The held-out recordings choose
  the best design and the best epoch; the test recordings never choose anything.
- `scaling_stats` computes, for every channel, the mean and the spread (standard deviation) of its
  counts on the fitting recordings. Most models subtract the mean and divide by the spread, so every
  channel has a similar range of values. These numbers are fixed, not trained.
- `input_statistics` measures how many spikes and non-empty cells a test recording has on average; the
  energy estimate needs them.
- `window_orders` makes a random order of the windows for every recording. Each recording gets its own
  order, so the order carries no information that a network could learn.
- `to_gpu` copies recordings to the GPU; `get_batch` takes a batch of them, each in its own window order
  when one is given; `reorder` makes a copy of a whole table with every recording's windows reordered.

In [ ]:
def holdout(y, share, seed):
    """Hold out round(share x class size) recordings of every class (at least one), chosen at random.
    Returns the rows kept for fitting and the held-out rows."""
    rng = np.random.default_rng(seed)                     # a random generator with a fixed seed
    held = []                                             # the held-out rows, class by class
    for k in range(K):                                    # every class
        rows_k = np.where(y == k)[0]                      # the recordings of class k
        n_k = max(1, int(round(share * len(rows_k))))     # how many of them to hold out
        held.append(rng.choice(rows_k, n_k, replace=False))   # choose them at random
    held = np.sort(np.concatenate(held))                  # all held-out rows, in increasing order
    fit = np.setdiff1d(np.arange(len(y)), held)           # every other row is used for fitting
    return fit, held


def scaling_stats(X):
    """Per-channel mean and standard deviation of the counts per window, and of the totals per recording."""
    totals = X.sum(axis=1).astype(np.float64)                                  # every recording's total per channel
    stats = {}
    stats['mu_window'] = X.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)           # mean count per window
    stats['sd_window'] = (X.std(axis=(0, 1), dtype=np.float64) + 1e-6).astype(np.float32)   # its spread (+ a tiny
    stats['mu_total'] = totals.mean(0).astype(np.float32)                                    # number, never 0)
    stats['sd_total'] = (totals.std(0) + 1e-6).astype(np.float32)                            # the same for totals
    return stats


def input_statistics(X):
    """Average size of one test recording: windows, channels, spikes, non-empty cells, active channels."""
    spikes, cells, active = 0, 0, 0
    for a in range(0, len(X), 1000):                      # 1,000 recordings at a time
        part = X[a:a + 1000]
        spikes += int(part.sum(dtype=np.int64))           # input spikes
        cells += int(np.count_nonzero(part))              # (window, channel) cells with at least one spike
        active += int(np.count_nonzero(part.sum(axis=1, dtype=np.int64)))   # channels with at least one spike
    n = len(X)
    return {'windows': int(X.shape[1]), 'channels': int(X.shape[2]), 'spikes': spikes / n,
            'nonzero_cells': cells / n, 'nonzero_channels': active / n}


def window_orders(n, T, seed, share=1.0):
    """A random order of the T windows for each of n recordings (an n x T array of window numbers).
    With share < 1 only that share of the windows moves; the others stay where they are."""
    rng = np.random.default_rng(seed)                     # a random generator with a fixed seed
    orders = np.tile(np.arange(T), (n, 1))                # start from the true order 0, 1, ..., T-1
    k = int(round(share * T))                             # how many windows of each recording move
    if k < 2:                                             # moving 0 or 1 window changes nothing
        return orders
    for i in range(n):                                    # every recording gets its own order
        chosen = rng.choice(T, k, replace=False)          # which windows move
        orders[i, chosen] = rng.permutation(chosen)       # they swap places among themselves at random
    return orders


def to_gpu(X, rows):
    """The recordings `rows` of table X as one tensor of float numbers on the GPU."""
    out = torch.empty((len(rows), X.shape[1], X.shape[2]), dtype=torch.float32, device=DEVICE)   # room on the GPU
    for a in range(0, len(rows), 4096):                   # copied 4,096 recordings at a time (saves memory)
        part = X[rows[a:a + 4096]].astype(np.float32)     # these recordings as float numbers
        out[a:a + 4096] = torch.from_numpy(part).to(DEVICE)
    return out


def orders_to_gpu(orders):
    """Window orders as a tensor on the GPU (or None when there are none)."""
    if orders is None:
        return None
    return torch.from_numpy(orders).to(DEVICE)


def get_batch(X, rows, order=None):
    """The recordings `rows` of a table on the GPU; with `order`, each one's windows in its own order."""
    rows = torch.as_tensor(rows, device=X.device)         # the row numbers, on the GPU
    xb = X[rows]                                          # these recordings: (batch, windows, channels)
    if order is not None:                                 # window t of the result is window o[t] of the recording
        o = order[rows]                                   # the window order of each recording in the batch
        xb = torch.gather(xb, 1, o[:, :, None].expand(-1, -1, xb.shape[2]))   # take the windows in that order
    return xb


def reorder(X, orders):
    """A copy of a whole table on the GPU with every recording's windows in its given order."""
    o = orders_to_gpu(orders)
    return torch.gather(X, 1, o[:, :, None].expand(-1, -1, X.shape[2]))


def thin(rows, n):
    """At most n of the rows, spread evenly (used only in a quick test)."""
    if len(rows) <= n:
        return rows
    return rows[np.linspace(0, len(rows) - 1, n).astype(np.int64)]

## Step 8. How energy is computed (computed, not measured)

No chip is measured. Energy per test recording follows the hardware-aware accounting of Dampfhoffer
et al. (2022): every weight and every neuron's state sits in on-chip memory, and every memory access
and every arithmetic operation costs energy. Memory accesses cost the most. Every model is charged on
the same imagined digital chip, with costs for 8-bit numbers in 45 nm technology based on Horowitz (2014):

| symbol | cost | what |
|---|---|---|
| m | 1.25 pJ | read or write one 8-bit number (a 64-bit read from an 8 KB memory, 10 pJ, divided by 8) |
| c | 0.23 pJ | one 8-bit multiply-add (multiplication 0.2 pJ + addition 0.03 pJ) |
| a | 0.03 pJ | one 8-bit addition |

From these, the cost of the operations the models do:

| operation | cost | pJ | what is charged |
|---|---|---|---|
| spike delivered to one synapse | 3m + a | 3.78 | read the weight; read and write the target neuron's state; one addition |
| first-layer multiply-add on a non-empty cell | 3m + c | 3.98 | read the weight; read and write the partial sum |
| any other multiply-add, no reuse | 4m + c | 5.23 | read the input and the weight; read and write the partial sum |
| multiply-add with ideal reuse | m + 4c | 2.17 | the weight is read once; the rest stays in registers |
| multiply-add, weights kept on chip | 4c + m/T | | the weight is read once per recording of T windows |
| spike with its weight in a register | 2m + c + a | 2.76 | |
| neuron state update | 2m + c | 2.73 | read the state, write it back, one multiply-add |
| spike added to a channel total | 2m + a | 2.53 | the count classifier |

Every model gets three numbers: **worst** (no data reuse), **within** (best case within a window: every
weight read once per window) and **across** (best case across the windows: the weights stay on the
chip for the whole recording). Models are compared within one case; a conclusion that holds in all
three cases does not depend on the hardware.

Per test recording: T windows, C channels, S input spikes, N non-empty (window, channel) cells.

**Conventional networks and order-blind designs.** W1 is the width of the first layer, P the
multiply-adds per window after the first layer, and Q the multiply-adds of a part that runs once per
recording. The input scaling folds into the first layer, so an empty cell costs only its read:

```
worst  = T C m + N W1 (3m + c) + (T P + Q)(4m + c)
within = T C m + (N W1 + T P + Q)(m + 4c)
across = T C m + (N W1 + T P)(4c + m / T) + Q (m + 4c)
```

**Spiking networks.** The first layer is charged either per input spike, S H (3m + a), or per
non-empty cell, N H (3m + c): the worst case takes the dearer of the two, the within case the cheaper
of S H (3m + a) and N H (m + 4c), the across case the cheaper of S H (2m + c + a) and N H (4c + m / T).
Every hidden spike costs 3m + a per synapse it reaches (2m + c + a across the windows), and every
neuron's state costs 2m + c per window.

In [ ]:
E_MEM = 1.25                                  # m: read or write one 8-bit number (pJ)
E_MAC = 0.23                                  # c: one multiply-add (pJ)
E_ADD = 0.03                                  # a: one addition (pJ)
COST_SPIKE = 3 * E_MEM + E_ADD                # a spike delivered to one synapse
COST_FIRST = 3 * E_MEM + E_MAC                # a first-layer multiply-add on a non-empty cell
COST_OTHER = 4 * E_MEM + E_MAC                # any other multiply-add without reuse
COST_REUSE = E_MEM + 4 * E_MAC                # a multiply-add with ideal reuse
COST_SPIKE_REG = 2 * E_MEM + E_MAC + E_ADD    # a spike whose weight is in a register
COST_UPDATE = 2 * E_MEM + E_MAC               # one neuron state update
COST_COUNT = 2 * E_MEM + E_ADD                # a spike added to a channel total
COSTS = {'memory_access': E_MEM, 'multiply_add': E_MAC, 'addition': E_ADD}   # saved with every experiment


def dense_energy(st, W1, P, Q):
    """Energy (pJ per recording) of a conventional network or order-blind design, in the three cases."""
    T, Cn, N = st['windows'], st['channels'], st['nonzero_cells']
    reads = T * Cn * E_MEM                                        # every input cell is read once
    kept = 4 * E_MAC + E_MEM / T                                  # a multiply-add with the weights kept on chip
    worst = reads + N * W1 * COST_FIRST + (T * P + Q) * COST_OTHER
    within = reads + (N * W1 + T * P + Q) * COST_REUSE
    across = reads + (N * W1 + T * P) * kept + Q * COST_REUSE
    parts = {'input_reads': reads, 'first_layer': N * W1 * COST_FIRST, 'other_layers': (T * P + Q) * COST_OTHER}
    return {'worst': worst, 'within': within, 'across': across, 'worst_is': 'no data reuse', 'parts': parts}


def spiking_energy(st, first_width, hidden_events, state_updates):
    """Energy (pJ per recording) of a spiking network.
    first_width: neurons every input spike reaches; hidden_events: hidden spikes x synapses each one
    reaches; state_updates: neuron states updated per recording."""
    T, S, N = st['windows'], st['spikes'], st['nonzero_cells']
    kept = 4 * E_MAC + E_MEM / T                                  # a multiply-add with the weights kept on chip
    per_spike = S * first_width * COST_SPIKE                      # first layer charged per input spike ...
    per_cell = N * first_width * COST_FIRST                       # ... or per non-empty cell
    hidden = hidden_events * COST_SPIKE                           # hidden spikes delivered
    state = state_updates * COST_UPDATE                           # neuron states updated
    worst = max(per_spike, per_cell) + hidden + state
    within = min(per_spike, N * first_width * COST_REUSE) + hidden + state
    across = min(S * first_width * COST_SPIKE_REG, N * first_width * kept) + hidden_events * COST_SPIKE_REG + state
    if per_spike >= per_cell:
        worst_is = 'per spike'
    else:
        worst_is = 'per cell'
    parts = {'first_layer_per_spike': per_spike, 'first_layer_per_cell': per_cell, 'hidden_spikes': hidden,
             'state_updates': state}
    return {'worst': worst, 'within': within, 'across': across, 'worst_is': worst_is, 'parts': parts}

## Step 9. The models

Every model takes a batch of tables, of shape (batch, T windows, 700 channels), and returns one score
per class; the class with the highest score is the answer. A model is built right after its seed is
set, and the order in which its layers are created fixes which random numbers start them, so a model
starts from the same weights whenever it is built with the same seed.

| model | kind | what it is |
|---|---|---|
| count_256 | counts only | channel totals -> 256 -> 20 classes |
| perminv | order-blind | every window 700 -> 128 -> 20, the scores of all windows added up |
| pool_128, pool_512 | order-blind | every window 700 -> 128 (or 512), averaged over windows, -> 128 -> 20 |
| meanmax_deep_128 | order-blind | every window 700 -> 128 -> 128, average and maximum over windows |
| ob_deep_1024_d10 | order-blind | the same idea at RadLIF's scale (1,024 units) |
| lif_std, rlif | spiking | 128 leaky integrate-and-fire neurons, without and with recurrent weights |
| rnn | conventional | a plain recurrent network with exactly rlif's number of weights |
| lstm_d0 to lstm_d70 | conventional | an LSTM of 128 cells at four levels of input dropout |
| radlif | spiking | RadLIF: two layers of 1,024 adaptive spiking neurons |

A name ending in `_d20`, `_d50` or `_d70` means dropout 0.2, 0.5 or 0.7 on the input while training
(dropout switches off a random share of the inputs, which helps a model generalise). Every model also
has an `energy` function that applies the formulas of step 8 to that model.

Most models start by scaling every channel: (count - mean) / spread, with the fixed numbers of
`scaling_stats`. They are stored with the model as "buffers": numbers that move to the GPU with the
model but are never trained.

### count_256: the floor

**Why.** It sees only each channel's total over the whole recording, so it loses both which channels
fire together within a moment and the order of the moments. Every other model should beat it.

**Architecture.** Channel totals, scaled -> 256 (ReLU) -> 20 classes.

**Energy.** Every input spike is added to its channel's total (2m + a); the 700 totals are read once;
the two layers run once per recording, on the non-empty totals.

In [ ]:
class CountClassifier(nn.Module):
    family = 'counts only'                                    # the kind of model (used by the analysis)

    def __init__(self, K, stats, h=256):
        super().__init__()                                    # the standard start of every PyTorch model
        self.net = nn.Sequential(nn.Linear(C, h), nn.ReLU(), nn.Linear(h, K))   # totals -> 256 -> K
        self.register_buffer('mu', torch.tensor(stats['mu_total']))           # mean total per channel
        self.register_buffer('sd', torch.tensor(stats['sd_total']))           # spread of the totals
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate

    def forward(self, x):                                     # x: (batch, windows, channels)
        totals = x.sum(1)                                     # add up the windows: one total per channel
        return self.net((totals - self.mu) / self.sd)         # scale the totals, then classify

    def energy(self, st, spikes=None, layer_spikes=None):
        reads = st['spikes'] * COST_COUNT + st['channels'] * E_MEM        # count every spike; read the totals
        macs1 = st['nonzero_channels'] * self.h                           # first layer, on non-empty totals
        macs2 = self.h * self.K                                           # second layer
        worst = reads + macs1 * COST_FIRST + macs2 * COST_OTHER           # no reuse
        best = reads + (macs1 + macs2) * COST_REUSE                       # runs once per recording: within and
        parts = {'input_reads': reads, 'first_layer': macs1 * COST_FIRST, 'other_layers': macs2 * COST_OTHER}
        return {'worst': worst, 'within': best, 'across': best, 'worst_is': 'no data reuse', 'parts': parts}   # across agree

### perminv: summed scores per window

**Why.** The simplest order-blind design, and the one with exactly lif_std's number of weights: it has
lif_std's two layers, but its units keep nothing from one window to the next. It is lif_std's
same-size partner.

**Architecture.** Every window: 700 -> 128 (ReLU) -> 20 scores. The scores of all windows are added up;
adding up does not depend on the order.

**Energy.** W1 = 128; P = 128 x 20 (the second layer runs in every window); Q = 0.

In [ ]:
class PerWindowScores(nn.Module):
    family = 'order-blind'

    def __init__(self, K, stats, h=H):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # mean count per channel and window
        self.register_buffer('sd', torch.tensor(stats['sd_window']))     # its spread
        self.net = nn.Sequential(nn.Linear(C, h), nn.ReLU(), nn.Linear(h, K))   # used on every window
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate

    def forward(self, x):
        x = (x - self.mu) / self.sd                           # scale every channel
        return self.net(x).sum(1)                             # scores of every window, added up

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=self.h, P=self.h * self.K, Q=0)

### pool_128 and pool_512: mean pooling

**Why.** The features of every window are averaged. pool_128 has exactly the weights of rlif and of
rnn (their features are averaged instead of carried from window to window); pool_512 has the LSTM's
number of weights within 0.1%. They are the same-size partners of those networks.

**Architecture.** Every window 700 -> h1 (ReLU); the average over windows; h1 -> 128 (ReLU) -> 20.

**Energy.** W1 = h1; P = 0; Q = h1 x 128 + 128 x 20.

In [ ]:
class MeanPool(nn.Module):
    family = 'order-blind'

    def __init__(self, K, stats, h1, h2):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.enc = nn.Sequential(nn.Linear(C, h1), nn.ReLU())            # every window -> h1 features
        self.dec = nn.Sequential(nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, K))   # the average -> classes
        self.h1, self.h2, self.K = h1, h2, K                  # sizes, remembered for the energy estimate

    def forward(self, x):
        features = self.enc((x - self.mu) / self.sd)          # (batch, windows, h1)
        return self.dec(features.mean(1))                     # average over the windows, then classify

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=self.h1, P=0, Q=self.h1 * self.h2 + self.h2 * self.K)

### meanmax_deep_128 (with input dropout: _d20, _d50, _d70)

**Why.** A deeper per-window encoder, pooled by average and maximum.

**Architecture.** Every window 700 -> 128 -> 128 (ReLU after each); average and maximum over windows
(256) -> 128 (ReLU) -> 20. Optional dropout p on the scaled input while training.

**Energy.** W1 = 128; P = 128 x 128; Q = 256 x 128 + 128 x 20.

In [ ]:
class MeanMaxDeep(nn.Module):
    family = 'order-blind'

    def __init__(self, K, stats, h=H, p_in=0.0):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.enc = nn.Sequential(nn.Linear(C, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())   # two layers per window
        self.dec = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, K))          # average and max -> classes
        if p_in > 0:
            self.drop = nn.Dropout(p_in)                      # dropout on the input while training
        else:
            self.drop = None
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate

    def forward(self, x):
        x = (x - self.mu) / self.sd                           # scale every channel
        if self.drop is not None:
            x = self.drop(x)
        f = self.enc(x)                                       # features of every window
        return self.dec(torch.cat([f.mean(1), f.max(1).values], 1))

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=self.h, P=self.h * self.h, Q=2 * self.h * self.h + self.h * self.K)

### ob_deep_1024_d10: an order-blind design at RadLIF's scale

**Why.** It has RadLIF's width (1,024), its depth (two layers per window), batch normalisation, dropout
0.1 and RadLIF's training recipe, but it is blind to order: the order-blind design of RadLIF's size
(0.16% fewer weights).

**Architecture.** Every window 700 -> 1024 -> 1024 (layers without bias, each followed by batch
normalisation over all recordings and windows, ReLU and dropout 0.1); average and maximum over windows
(2048) -> 1024 (batch normalisation, ReLU, dropout) -> 20. Batch normalisation rescales every unit
using the mean and spread of the current batch, which makes training of large networks more stable.

**Energy.** W1 = 1024; P = 1024 x 1024; Q = 2048 x 1024 + 1024 x 20 (batch normalisation folds into the
weights when testing).

In [ ]:
class TimeBatchNorm(nn.Module):
    """Batch normalisation over every recording and window together; it does not depend on the order."""

    def __init__(self, h):
        super().__init__()
        self.bn = nn.BatchNorm1d(h, momentum=0.05)            # PyTorch's batch normalisation for h units

    def forward(self, x):                                     # x: (batch, windows, h)
        flat = x.reshape(-1, x.shape[-1])                     # all windows of all recordings in one long list
        return self.bn(flat).reshape(x.shape)                 # normalise, then back to (batch, windows, h)


class MeanMaxDeepBN(nn.Module):
    family = 'order-blind'

    def __init__(self, K, stats, h=1024, p=0.1):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.enc = nn.Sequential(nn.Linear(C, h, bias=False), TimeBatchNorm(h), nn.ReLU(), nn.Dropout(p),   # layer 1
                                 nn.Linear(h, h, bias=False), TimeBatchNorm(h), nn.ReLU(), nn.Dropout(p))   # layer 2
        self.dec = nn.Sequential(nn.Linear(2 * h, h, bias=False), nn.BatchNorm1d(h, momentum=0.05), nn.ReLU(),
                                 nn.Dropout(p), nn.Linear(h, K))                                            # head
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate

    def forward(self, x):
        f = self.enc((x - self.mu) / self.sd)                 # features of every window
        return self.dec(torch.cat([f.mean(1), f.max(1).values], 1))

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=self.h, P=self.h * self.h, Q=2 * self.h * self.h + self.h * self.K)

### lif_std and rlif: small spiking networks

**Why.** The small spiking networks of the study. lif_std's only memory is each neuron's leaky voltage;
rlif adds recurrent weights among its neurons. Together with rnn (the same weights in a conventional
recurrent network) they make spiking and conventional twins of equal size.

**Architecture.** 128 leaky integrate-and-fire neurons. In every window, in order:

```
voltage = beta x voltage + input weights x counts [+ recurrent weights x last spikes] - last spike x 1
spike   = 1 if voltage > 1, else 0
```

A neuron that spiked loses 1 from its voltage in the next window. A readout (128 -> 20) is applied to
every window's spikes, and the results are added up. beta = 0.9 per window at 25 windows, and the same
memory in milliseconds for finer windows. This is snnTorch 1.0.0's Leaky and RLeaky neuron written out
step by step (the self-checks confirm that the two agree exactly).

A spike is a jump from 0 to 1, whose slope is zero almost everywhere, so training uses a smooth
stand-in for the slope: the slope of a "fast sigmoid", 1 / (25 |x| + 1)^2, as snnTorch does.

**Energy.** The first layer per input spike or per non-empty cell (128 neurons reached); every hidden
spike reaches the 20 readout units (and, in rlif, the 128 neurons of the layer); 128 state updates per window.

In [ ]:
class SpikeFastSigmoid(torch.autograd.Function):
    """A spike (1) where the input is above 0, else 0; when training, the slope of a fast sigmoid."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)                              # remember the input for the backward pass
        return (x > 0).float()                                # 1 where above the threshold, else 0

    @staticmethod
    def backward(ctx, grad):
        (x,) = ctx.saved_tensors
        return grad / (25 * torch.abs(x) + 1.0) ** 2          # the smooth stand-in slope


class LIFNet(nn.Module):
    family = 'spiking'

    def __init__(self, K, beta, stats, recurrent=False, h=H):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.fc1 = nn.Linear(C, h)                            # input weights: 700 channels -> 128 neurons
        self.fc2 = nn.Linear(h, K)                            # readout: 128 neurons -> 20 classes
        if recurrent:
            self.recurrent = nn.Linear(h, h)                  # rlif only: weights from the neurons to themselves
        else:
            self.recurrent = None
        self.register_buffer('beta', torch.as_tensor(beta, dtype=torch.float32))   # share of voltage kept
        self.register_buffer('threshold', torch.as_tensor(1.0))                     # spike when above 1
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate
        self.counting = False                                 # True while testing: count the hidden spikes
        self.spike_total = 0.0                                # hidden spikes counted so far

    def forward(self, x):
        x = (x - self.mu) / self.sd                           # scale every channel
        beta = self.beta.clamp(0, 1)                          # beta must stay between 0 and 1
        mem = torch.zeros(x.shape[0], self.h, device=x.device)   # every neuron's voltage, starting at 0
        spk = torch.zeros_like(mem)                           # every neuron's spike in the last window
        out = 0                                               # the class scores, added up over windows
        n_spikes = 0                                          # hidden spikes of this batch
        for t in range(x.shape[1]):                           # the windows IN ORDER: here order can matter
            reset = spk.detach()                              # neurons that spiked in the last window lose 1
            if self.recurrent is None:
                mem = beta * mem + self.fc1(x[:, t]) - reset * self.threshold
            else:
                mem = beta * mem + self.fc1(x[:, t]) + self.recurrent(spk) - reset * self.threshold
            spk = SpikeFastSigmoid.apply(mem - self.threshold)   # spike where the voltage is above 1
            out = out + self.fc2(spk)                         # readout of this window, added up
            if self.counting:
                n_spikes = n_spikes + spk.detach().sum()      # count this window's spikes
        if self.counting:
            self.spike_total += float(n_spikes)
        return out

    def energy(self, st, spikes, layer_spikes=None):
        fan_out = self.K                                      # every hidden spike reaches the 20 readout units
        if self.recurrent is not None:
            fan_out = fan_out + self.h                        # ... and, in rlif, the 128 neurons of the layer
        return spiking_energy(st, first_width=self.h, hidden_events=spikes * fan_out,
                              state_updates=self.h * st['windows'])   # one state per neuron per window

### rnn and the LSTMs (lstm_d0, lstm_d20, lstm_d50, lstm_d70)

**Why.** Conventional networks that read the windows in order. rnn (tanh units) has exactly rlif's
number of weights: its conventional twin. The LSTM (128 cells with gated memory) is the strongest
conventional network here; its input dropout is chosen on the held-out recordings, like the designs'.

**Architecture.** Dropout p on the scaled input; PyTorch's RNN or LSTM, 700 -> 128; a readout 128 -> 20
at every window, added up over the windows.

**Energy.** rnn: W1 = 128, P = 128 x 128 + 128 x 20. LSTM: W1 = 4 x 128 (four gates), P = 4 x 128 x 128 +
128 x 20. Q = 0.

In [ ]:
class RecurrentNet(nn.Module):
    family = 'conventional'

    def __init__(self, kind, K, stats, p_in=0.0, h=H):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.drop = nn.Dropout(p_in)                          # dropout on the input (p_in = 0: none)
        if kind == 'rnn':
            self.rec = nn.RNN(C, h, batch_first=True)         # a plain recurrent layer (tanh)
        else:
            self.rec = nn.LSTM(C, h, batch_first=True)        # an LSTM layer
        self.fc = nn.Linear(h, K)                             # readout
        self.kind, self.h, self.K = kind, h, K                # remembered for the energy estimate

    def forward(self, x):
        x = self.drop((x - self.mu) / self.sd)                # scale, then dropout (only while training)
        hidden, _ = self.rec(x)                               # the hidden state after every window, in order
        return self.fc(hidden).sum(1)                         # readout of every window, added up

    def energy(self, st, spikes=None, layer_spikes=None):
        if self.kind == 'rnn':
            gates = 1
        else:
            gates = 4                                         # an LSTM has four gates
        return dense_energy(st, W1=gates * self.h, P=gates * self.h * self.h + self.h * self.K, Q=0)

### RadLIF: the building blocks

The spike function, one layer of adaptive spiking neurons with recurrent weights, and the
non-spiking readout. This code is adapted from SpArch (BSD-3-Clause licence; the notice is kept in the
cell). In every window, for every neuron:

```
adaptation w = beta w + a u + b s
voltage    u = alpha (u - s) + (1 - alpha) (input + recurrent input - w)
spike      s = 1 if u > 1, else 0
```

The constants alpha, beta, a and b are learned for every neuron, within fixed limits. Every pass starts
from random states.

In [ ]:
# SpikeFunctionBoxcar, RadLIFLayer, ReadoutLayer and RadLIFNet are adapted from SpArch,
# file sparch/models/snns.py (https://github.com/idiap/sparch). The network and its update are kept
# as in that code; the classes here also count hidden spikes for the energy estimate.
#
# Copyright (c) 2022 Idiap Research Institute <contact@idiap.ch>. Contributor: Alexandre Bittar.
#
# Redistribution and use in source and binary forms, with or without modification, are permitted
# provided that the following conditions are met:
#
# 1. Redistributions of source code must retain the above copyright notice, this list of conditions
#    and the following disclaimer.
#
# 2. Redistributions in binary form must reproduce the above copyright notice, this list of
#    conditions and the following disclaimer in the documentation and/or other materials provided
#    with the distribution.
#
# 3. Neither the name of the copyright holder nor the names of its contributors may be used to
#    endorse or promote products derived from this software without specific prior written
#    permission.
#
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND ANY EXPRESS OR
# IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND
# FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR
# CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE,
# DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER
# IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT
# OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

class SpikeFunctionBoxcar(torch.autograd.Function):
    """A spike where the input (voltage minus threshold) is above 0; when training, the slope passes
    only inside a box of width 1 around the threshold."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)                              # remember the input for the backward pass
        return x.gt(0).to(x.dtype)                            # 1 above the threshold, else 0

    @staticmethod
    def backward(ctx, grad_spikes):
        (x,) = ctx.saved_tensors
        grad_x = grad_spikes.clone()
        grad_x[x <= -0.5] = 0                                 # no slope far below the threshold
        grad_x[x > 0.5] = 0                                   # no slope far above it
        return grad_x


class RadLIFLayer(nn.Module):
    """One layer of adaptive leaky integrate-and-fire neurons with recurrent weights inside the layer."""

    def __init__(self, input_size, hidden_size, dropout=0.0, threshold=1.0):
        super().__init__()
        self.threshold = threshold                            # spike when the voltage is above 1
        self.alpha_lim = [np.exp(-1 / 5), np.exp(-1 / 25)]   # voltage memory: 5 to 25 windows
        self.beta_lim = [np.exp(-1 / 30), np.exp(-1 / 120)]  # adaptation memory: 30 to 120 windows
        self.a_lim = [-1.0, 1.0]                              # limits of a
        self.b_lim = [0.0, 2.0]                               # limits of b
        self.W = nn.Linear(input_size, hidden_size, bias=False)    # input weights
        self.V = nn.Linear(hidden_size, hidden_size, bias=False)   # recurrent weights
        self.alpha = nn.Parameter(torch.Tensor(hidden_size))       # one learned constant per neuron
        self.beta = nn.Parameter(torch.Tensor(hidden_size))
        self.a = nn.Parameter(torch.Tensor(hidden_size))
        self.b = nn.Parameter(torch.Tensor(hidden_size))
        nn.init.uniform_(self.alpha, self.alpha_lim[0], self.alpha_lim[1])   # random start within the limits
        nn.init.uniform_(self.beta, self.beta_lim[0], self.beta_lim[1])
        nn.init.uniform_(self.a, self.a_lim[0], self.a_lim[1])
        nn.init.uniform_(self.b, self.b_lim[0], self.b_lim[1])
        nn.init.orthogonal_(self.V.weight)                    # recurrent weights start orthogonal
        self.norm = nn.BatchNorm1d(hidden_size, momentum=0.05)    # batch normalisation of the input current
        self.drop = nn.Dropout(p=dropout)                     # dropout on the layer's spikes

    def forward(self, x):
        Wx = self.W(x)                                        # input current of every neuron in every window
        Wx = self.norm(Wx.reshape(Wx.shape[0] * Wx.shape[1], Wx.shape[2])).reshape(Wx.shape)   # normalised
        return self.drop(self.radlif_cell(Wx))                # the spikes, with dropout while training

    def radlif_cell(self, Wx):
        ut = torch.rand(Wx.shape[0], Wx.shape[2]).to(device=Wx.device, dtype=Wx.dtype)   # random starting voltage,
        wt = torch.rand(Wx.shape[0], Wx.shape[2]).to(device=Wx.device, dtype=Wx.dtype)   # adaptation and spikes
        st = torch.rand(Wx.shape[0], Wx.shape[2]).to(device=Wx.device, dtype=Wx.dtype)   # (from the CPU's generator)
        out = []
        alpha = torch.clamp(self.alpha, min=self.alpha_lim[0], max=self.alpha_lim[1])    # keep the learned
        beta = torch.clamp(self.beta, min=self.beta_lim[0], max=self.beta_lim[1])        # constants within
        a = torch.clamp(self.a, min=self.a_lim[0], max=self.a_lim[1])                    # their limits
        b = torch.clamp(self.b, min=self.b_lim[0], max=self.b_lim[1])
        V = self.V.weight.clone().fill_diagonal_(0)           # no neuron feeds itself
        for t in range(Wx.shape[1]):                          # the windows in order
            wt = beta * wt + a * ut + b * st                  # adaptation
            ut = alpha * (ut - st) + (1 - alpha) * (Wx[:, t, :] + torch.matmul(st, V) - wt)   # voltage
            st = SpikeFunctionBoxcar.apply(ut - self.threshold)                              # spikes
            out.append(st)
        return torch.stack(out, dim=1)                        # spikes of every window: (batch, windows, neurons)


class ReadoutLayer(nn.Module):
    """Non-spiking leaky units, one per class; the scores are their softmax added up over windows."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.alpha_lim = [np.exp(-1 / 5), np.exp(-1 / 25)]   # memory of 5 to 25 windows
        self.W = nn.Linear(input_size, hidden_size, bias=False)    # hidden spikes -> class units
        self.alpha = nn.Parameter(torch.Tensor(hidden_size))
        nn.init.uniform_(self.alpha, self.alpha_lim[0], self.alpha_lim[1])
        self.norm = nn.BatchNorm1d(hidden_size, momentum=0.05)

    def forward(self, x):
        Wx = self.W(x)
        Wx = self.norm(Wx.reshape(Wx.shape[0] * Wx.shape[1], Wx.shape[2])).reshape(Wx.shape)
        ut = torch.rand(Wx.shape[0], Wx.shape[2]).to(device=Wx.device, dtype=Wx.dtype)   # random start
        out = torch.zeros(Wx.shape[0], Wx.shape[2], device=Wx.device, dtype=Wx.dtype)
        alpha = torch.clamp(self.alpha, min=self.alpha_lim[0], max=self.alpha_lim[1])
        for t in range(Wx.shape[1]):                          # the windows in order
            ut = alpha * ut + (1 - alpha) * Wx[:, t, :]       # leaky units
            out = out + F.softmax(ut, dim=1)                  # this window's class probabilities, added up
        return out

### radlif: RadLIF, two layers of 1,024 adaptive spiking neurons

**Why.** A published spiking network at full scale (94.62% on SHD in its authors' paper, Bittar and
Garner 2022), the network here most likely to show that order is needed. It is trained with its
authors' recipe: Adam at 0.01, the learning rate x 0.7 after 10 epochs without a better held-out
accuracy, dropout 0.1, batch normalisation, 60 epochs, the best epoch kept.

**Architecture.** Raw counts (no scaling, as in its authors' code) -> RadLIF layer -> RadLIF layer ->
leaky readout, with dropout 0.1 on each hidden layer's spikes.

**Energy.** Every input spike reaches the 1,024 neurons of layer 1; a spike of layer 1 reaches layer 2
(1,024) and its own layer (1,023: no neuron feeds itself); a spike of layer 2 reaches its own layer
(1,023) and the 20 readout units; two state variables (voltage, adaptation) per neuron per window.

In [ ]:
class RadLIFNet(nn.Module):
    family = 'spiking'

    def __init__(self, K, hidden=1024, n_hidden=2, dropout=0.1):
        super().__init__()
        sizes = [C] + [hidden] * n_hidden                     # 700 -> 1024 -> 1024
        layers = []
        for i in range(n_hidden):
            layers.append(RadLIFLayer(sizes[i], sizes[i + 1], dropout))   # one RadLIF layer
        self.layers = nn.ModuleList(layers)                   # the hidden layers, as one list
        self.readout = ReadoutLayer(hidden, K)                # the class units
        self.h, self.K = hidden, K                            # sizes, remembered for the energy estimate
        self.counting = False                                 # True while testing: count the hidden spikes
        self.spike_total = 0.0
        self.layer_spikes = [0.0] * n_hidden                  # hidden spikes of every layer

    def forward(self, x):
        for k in range(len(self.layers)):
            x = self.layers[k](x)                             # the spikes of hidden layer k
            if self.counting:
                n = float(x.sum())                            # count them
                self.spike_total += n
                self.layer_spikes[k] += n
        return self.readout(x)

    def tau_quartiles(self):
        """Quartiles of each hidden layer's learned voltage time constants, in windows (-1 / ln alpha)."""
        result = []
        for layer in self.layers:
            alpha = layer.alpha.detach().clamp(layer.alpha_lim[0], layer.alpha_lim[1]).cpu().double().numpy()
            result.append(np.percentile(-1 / np.log(alpha), [25, 50, 75]).tolist())
        return result

    def energy(self, st, spikes=None, layer_spikes=None):
        s1, s2 = layer_spikes                                 # hidden spikes of layer 1 and of layer 2
        events = s1 * (self.h + (self.h - 1)) + s2 * ((self.h - 1) + self.K)   # x synapses each spike reaches
        states = len(self.layers) * self.h * st['windows'] * 2               # 2 states per neuron per window
        return spiking_energy(st, first_width=self.h, hidden_events=events, state_updates=states)

### Building a model by its name

`make_model` builds a fresh, untrained model from its name. The number of classes K, the neurons' beta
and the scaling numbers are given to it, because they differ between the parts of this notebook.

In [ ]:
def make_model(name, K, beta, stats):
    if name == 'count_256':
        return CountClassifier(K, stats, 256)
    if name == 'perminv':
        return PerWindowScores(K, stats, H)
    if name == 'pool_128':
        return MeanPool(K, stats, H, H)
    if name == 'pool_512':
        return MeanPool(K, stats, 4 * H, H)
    if name == 'meanmax_deep_128':
        return MeanMaxDeep(K, stats, H, 0.0)
    if name in ('meanmax_deep_128_d20', 'meanmax_deep_128_d50', 'meanmax_deep_128_d70'):
        return MeanMaxDeep(K, stats, H, int(name[-2:]) / 100)     # the last two digits: d20 -> dropout 0.2
    if name == 'ob_deep_1024_d10':
        return MeanMaxDeepBN(K, stats, 1024, 0.1)
    if name == 'lif_std':
        return LIFNet(K, beta, stats, recurrent=False)
    if name == 'rlif':
        return LIFNet(K, beta, stats, recurrent=True)
    if name == 'rnn':
        return RecurrentNet('rnn', K, stats, 0.0)
    if name == 'lstm':
        return RecurrentNet('lstm', K, stats, 0.0)
    if name in ('lstm_d0', 'lstm_d20', 'lstm_d50', 'lstm_d70'):
        return RecurrentNet('lstm', K, stats, int(name[len('lstm_d'):]) / 100)   # lstm_d50 -> dropout 0.5
    if name == 'radlif':
        return RadLIFNet(K)
    raise ValueError('unknown model: ' + name)


def n_weights(model):
    """The number of trainable numbers (weights and biases) of a model."""
    return sum(p.numel() for p in model.parameters())

## Step 10. Training and testing

**Training.** A fresh model sees the fitting recordings in batches of 128, in a new random order of the
recordings every epoch (never of the windows). After each batch the optimiser changes the weights a
little to lower the cross-entropy loss (a measure of how wrong the answers are). When the recipe keeps
the best epoch, the model is tested on the held-out recordings after every epoch, and the weights of
the best epoch are kept.

**Testing.** The model answers every recording; the accuracy is the share of right answers. For
spiking networks the hidden spikes are counted, because the energy estimate needs them. RadLIF starts
every test from random states; a fixed seed makes them the same every time, without disturbing the
random numbers of training.

In [ ]:
def make_optimizer(model, recipe):
    """The optimiser of a recipe: Adam or Adamax, with the recipe's learning rate."""
    if recipe['optimizer'] == 'adamax':
        return torch.optim.Adamax(model.parameters(), lr=recipe['lr'])
    return torch.optim.Adam(model.parameters(), lr=recipe['lr'])


def test_model(model, X, y, order, seed):
    """Accuracy, the answers, and (spiking networks) the hidden spikes per recording, in all and per layer."""
    model.eval()                                              # testing mode: no dropout, fixed batch statistics
    if model.family == 'spiking':                             # start counting the hidden spikes
        model.counting = True
        model.spike_total = 0.0
        if hasattr(model, 'layer_spikes'):                    # RadLIF also counts per layer
            model.layer_spikes = [0.0] * len(model.layer_spikes)
    answers = []
    with torch.no_grad():                                     # testing only: slopes are not needed
        with torch.random.fork_rng(devices=[]):               # remember the CPU's random state ...
            torch.default_generator.manual_seed(seed)         # ... use a fixed seed for this test ...
            for i in range(0, len(y), BATCH):                 # ... (the old state comes back afterwards)
                rows = np.arange(i, min(i + BATCH, len(y)))   # the next 128 recordings
                scores = model(get_batch(X, rows, order))     # one score per class
                answers.append(scores.argmax(1).cpu().numpy())    # the class with the highest score
    answers = np.concatenate(answers)                         # all answers in one array
    spikes, layer_spikes = None, None
    if model.family == 'spiking':
        model.counting = False
        spikes = model.spike_total / len(y)                   # hidden spikes per recording
        if hasattr(model, 'layer_spikes'):
            layer_spikes = [v / len(y) for v in model.layer_spikes]
    accuracy = float((answers == y).mean())                   # share of right answers
    return accuracy, answers, spikes, layer_spikes


def train_model(model, X, y, order, recipe, X_val=None, y_val=None, order_val=None, eval_seed=0, label=''):
    """Train a model by its recipe. Returns the held-out accuracy after every epoch (when the recipe keeps
    the best epoch) and the epoch kept."""
    optimizer = make_optimizer(model, recipe)
    scheduler = None
    if recipe.get('plateau'):                                 # RadLIF's recipe: learning rate x 0.7 after
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7,    # 10 epochs
                                                               patience=10, min_lr=1e-6)             # without progress
    loss_fn = nn.CrossEntropyLoss()                           # how wrong the answers are
    best_acc, best_epoch, best_weights, curve = -1.0, None, None, []
    bar = tqdm(range(recipe['epochs']), desc=label, leave=False, unit='epoch')   # a progress bar over epochs
    for epoch in bar:
        model.train()                                         # training mode (dropout on)
        shuffled = np.random.permutation(len(y))              # a new random order of the recordings
        for i in range(0, len(shuffled), BATCH):
            rows = shuffled[i:i + BATCH]                      # the recordings of this batch
            xb = get_batch(X, rows, order)                    # their tables
            yb = torch.from_numpy(y[rows]).to(DEVICE)         # their classes
            optimizer.zero_grad()                             # forget the slopes of the last batch
            loss = loss_fn(model(xb), yb)                     # how wrong the model is on this batch
            loss.backward()                                   # the slope of the loss for every weight
            optimizer.step()                                  # change the weights a little
        if recipe['keep'] == 'best':                          # test on the held-out recordings
            acc_val = test_model(model, X_val, y_val, order_val, eval_seed)[0]
            curve.append(acc_val)
            if scheduler is not None:
                scheduler.step(acc_val)                       # lower the learning rate if progress stalls
            if acc_val > best_acc:                            # the best epoch so far: keep a copy of its weights
                best_acc, best_epoch = acc_val, epoch + 1
                best_weights = {}
                for key, value in model.state_dict().items():
                    best_weights[key] = value.detach().clone()
            bar.set_postfix(val=round(acc_val, 3), best_epoch=best_epoch)
    if best_weights is not None:
        model.load_state_dict(best_weights)                   # go back to the best epoch's weights
    return curve, best_epoch

## Step 11. Self-checks

Quick tests that stop the notebook if something is wrong:

- every order-blind model gives exactly the same answer when the windows are shuffled;
- perminv has exactly lif_std's number of weights, and pool_128 exactly rlif's and rnn's;
- lif_std and rlif give the same weights, scores, spike counts and slopes as snnTorch's Leaky and RLeaky;
- RadLIF gives the same scores as an independent version written with NumPy.

In [ ]:
def check_order_blind(names, K, beta, stats, x, x_shuffled):
    """Order-blind models must ignore the order of the windows; same-size partners must match in size."""
    sizes = {}
    for name in names:
        torch.manual_seed(0)
        model = make_model(name, K, beta, stats).to(DEVICE)
        with torch.no_grad():
            model.train()
            model(x)                                          # one training pass fills batch statistics
            model.eval()
            torch.manual_seed(1)
            out = model(x)                                    # scores with the windows in order
            torch.manual_seed(1)
            same = bool(torch.allclose(out, model(x_shuffled), atol=1e-4))   # ... and shuffled
        sizes[name] = n_weights(model)
        if model.family in ('order-blind', 'counts only'):
            assert same, name + ' should ignore the order of the windows'
    for a, b in [('perminv', 'lif_std'), ('pool_128', 'rlif'), ('pool_128', 'rnn')]:
        if a in sizes and b in sizes:
            assert sizes[a] == sizes[b], a + ' and ' + b + ' should have the same number of weights'
    print('   the order-blind models ignore the window order; the same-size partners match')


def check_against_snntorch(K, beta, stats, x, y):
    """lif_std and rlif against snnTorch's Leaky and RLeaky, built after the same seed (on the CPU)."""
    import snntorch as snn
    from snntorch import surrogate
    xc, yc = x.cpu(), torch.as_tensor(y).cpu()
    for recurrent in (False, True):
        torch.manual_seed(0)
        fc1, fc2 = nn.Linear(C, H), nn.Linear(H, K)           # snnTorch's version, layer by layer
        if recurrent:
            lif = snn.RLeaky(beta=beta, linear_features=H, spike_grad=surrogate.fast_sigmoid())
        else:
            lif = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid())
        torch.manual_seed(0)
        own = LIFNet(K, beta, stats, recurrent=recurrent)     # this notebook's version
        xs = (xc - torch.tensor(stats['mu_window'])) / torch.tensor(stats['sd_window'])
        out_ref, spikes_ref = 0, 0.0
        if recurrent:
            spk, mem = lif.init_rleaky()
        else:
            mem = lif.init_leaky()
        for t in range(xs.shape[1]):                          # snnTorch, window by window
            if recurrent:
                spk, mem = lif(fc1(xs[:, t]), spk, mem)
            else:
                spk, mem = lif(fc1(xs[:, t]), mem)
            out_ref = out_ref + fc2(spk)
            spikes_ref += spk.sum().item()
        own.counting = True
        out_own = own(xc)                                     # this notebook's version
        w_ref = [fc1.weight, fc1.bias, fc2.weight, fc2.bias]
        w_own = [own.fc1.weight, own.fc1.bias, own.fc2.weight, own.fc2.bias]
        if recurrent:
            w_ref += [lif.recurrent.weight, lif.recurrent.bias]
            w_own += [own.recurrent.weight, own.recurrent.bias]
        same_w = all(torch.equal(a, b) for a, b in zip(w_ref, w_own))            # same weights?
        nn.CrossEntropyLoss()(out_ref, yc).backward()
        nn.CrossEntropyLoss()(out_own, yc).backward()
        same_g = all(torch.equal(a.grad, b.grad) for a, b in zip(w_ref, w_own))  # same slopes?
        ok = same_w and torch.equal(out_ref, out_own) and spikes_ref == own.spike_total and same_g
        if recurrent:
            name = 'rlif'
        else:
            name = 'lif_std'
        print(f'   {name} against snnTorch {snn.__version__}: identical weights, scores, spike counts and slopes: {ok}')
        assert ok, 'the spiking neurons here differ from snnTorch 1.0.0'


def radlif_numpy(m, x, init):
    """RadLIF's testing pass written independently with NumPy, in double precision."""
    h = x.detach().cpu().double().numpy()
    k = 0
    for L in m.layers:
        z = h @ L.W.weight.detach().double().numpy().T                        # input current
        mean, var = L.norm.running_mean.double().numpy(), L.norm.running_var.double().numpy()
        gamma, shift = L.norm.weight.detach().double().numpy(), L.norm.bias.detach().double().numpy()
        z = (z - mean) / np.sqrt(var + L.norm.eps) * gamma + shift            # batch normalisation
        al = np.clip(L.alpha.detach().double().numpy(), L.alpha_lim[0], L.alpha_lim[1])
        be = np.clip(L.beta.detach().double().numpy(), L.beta_lim[0], L.beta_lim[1])
        aa = np.clip(L.a.detach().double().numpy(), L.a_lim[0], L.a_lim[1])
        bb = np.clip(L.b.detach().double().numpy(), L.b_lim[0], L.b_lim[1])
        V = L.V.weight.detach().double().numpy().copy()
        np.fill_diagonal(V, 0)
        u, w, sp = init[k], init[k + 1], init[k + 2]                          # the same random start
        k += 3
        outs = []
        for t in range(z.shape[1]):
            w = be * w + aa * u + bb * sp
            u = al * (u - sp) + (1 - al) * (z[:, t] + sp @ V - w)
            sp = (u - L.threshold > 0).astype(np.float64)
            outs.append(sp)
        h = np.stack(outs, 1)
    R = m.readout
    z = h @ R.W.weight.detach().double().numpy().T
    mean, var = R.norm.running_mean.double().numpy(), R.norm.running_var.double().numpy()
    gamma, shift = R.norm.weight.detach().double().numpy(), R.norm.bias.detach().double().numpy()
    z = (z - mean) / np.sqrt(var + R.norm.eps) * gamma + shift
    al = np.clip(R.alpha.detach().double().numpy(), R.alpha_lim[0], R.alpha_lim[1])
    u, out = init[k], np.zeros((z.shape[0], z.shape[2]))
    for t in range(z.shape[1]):
        u = al * u + (1 - al) * z[:, t]
        e = np.exp(u - u.max(1, keepdims=True))
        out = out + e / e.sum(1, keepdims=True)                               # softmax, added up
    return out


def check_radlif(K, x):
    """A small RadLIF (64 neurons per layer) must agree with the NumPy version."""
    torch.manual_seed(0)
    m = RadLIFNet(K, hidden=64).double().eval()
    xs = x[:3].cpu().double()
    torch.manual_seed(123)
    out = m(xs).detach().numpy()                              # PyTorch version
    torch.manual_seed(123)                                    # the same random starting states for NumPy
    init = [torch.rand(3, 64).double().numpy() for _ in range(6)] + [torch.rand(3, K).double().numpy()]
    diff = float(np.abs(out - radlif_numpy(m, xs, init)).max())
    print(f'   RadLIF against the NumPy version: largest difference {diff:.1e}')
    assert diff < 1e-9

## Step 12. Saving results and estimating the time

Every run is saved as one small file the moment it ends. It is written under a temporary name and then
renamed, so a file is either complete or absent. The time estimate times a few training steps of each
model on this machine: minutes per run = seconds per step x steps per epoch x epochs x 1.15 (testing
adds about 15%).

In [ ]:
def save_json(path, obj):
    """Save a dictionary as a .json file (written in one step)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + '.tmp', 'w') as f:
        json.dump(obj, f)
    os.replace(path + '.tmp', path)


def gpu_name():
    if DEVICE == 'cuda':
        return torch.cuda.get_device_name(0)
    return 'CPU'


def minutes_per_run(name, K, beta, stats, X, y):
    """Estimated minutes for one run of a model: a few training steps are timed on real batches."""
    recipe = recipe_of(name)
    torch.manual_seed(0)
    model = make_model(name, K, beta, stats).to(DEVICE)
    model.train()
    optimizer = make_optimizer(model, recipe)
    loss_fn = nn.CrossEntropyLoss()
    for step in range(4):                                     # step 0 warms up; steps 1 to 3 are timed
        if step == 1:
            if DEVICE == 'cuda':
                torch.cuda.synchronize()                      # wait until the GPU has finished
            start = time.time()
        rows = np.random.default_rng(step % 2).choice(len(y), min(BATCH, len(y)), replace=False)
        optimizer.zero_grad()
        loss_fn(model(get_batch(X, rows)), torch.from_numpy(y[rows]).to(DEVICE)).backward()
        optimizer.step()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    seconds_per_step = (time.time() - start) / 3
    steps_per_epoch = int(np.ceil(len(y) / BATCH))
    return seconds_per_step * steps_per_epoch * recipe['epochs'] * 1.15 / 60


def print_estimate(minutes, runs):
    """Print the estimate: minutes per run and runs still to do, model by model."""
    total = 0.0
    for name in minutes:
        hours = minutes[name] * runs[name] / 60
        total += hours
        print(f'   {name:24s} {minutes[name]:7.1f} min per run x {runs[name]} runs = {hours:6.2f} h')
    print(f'   estimated total: about {total:.1f} hours on {gpu_name()}')


# the overall progress bar of a part counts estimated minutes
BAR_FORMAT = '{l_bar}{bar}| {n:.1f}/{total:.1f} estimated min [{elapsed} elapsed, about {remaining} left]'

## Step 13. Count the spikes of every recording: 25 windows

Parts 1 and 2 use the table with 25 windows per recording. Counting takes about a minute.

In [ ]:
if RUN_PART_1 or RUN_PART_2:
    X25_train, y_train, spk_train, dur_train = count_spikes(SHD_TRAIN, 'relative', 25)
    X25_test, y_test, spk_test, dur_test = count_spikes(SHD_TEST, 'relative', 25)
    print('training table:', X25_train.shape, '| test table:', X25_test.shape)

## Part 1. The control tasks: answers known in advance

Three tasks made from SHD at 25 windows check that the method can find order when it is there, and
finds nothing when it is not:

- **chains**: three recordings of one speaker saying "one", "two" and "three" are joined, in all six
  orders (75 windows). The class is the order. Every order holds exactly the same windows, so an
  order-blind model must score exactly 1/6 and give one answer for all six chains of a triple.
- **reversal**: recordings as recorded (class 0) and with their windows reversed (class 1). Both hold
  the same windows, so an order-blind model must score exactly 1/2.
- **negative control**: the ordinary 20-word task, but every recording's windows are shuffled before
  anything else. Shuffling them once more cannot remove information that is no longer there, so a
  network's accuracy should not drop.

Training recordings make the training samples, test recordings the test samples. Nothing is held out,
because nothing is chosen here.

In [ ]:
def make_chains(X, y, spk, seed):
    """For every speaker: triples of recordings of 'one', 'two' and 'three', joined in all six orders."""
    rng = np.random.default_rng(seed)
    orders = list(permutations(range(3)))                     # (0,1,2), (0,2,1), ... the six orders
    X_out, y_out, groups, speakers = [], [], [], []
    group = 0
    for s in np.unique(spk):                                  # every speaker
        pools = []
        for word in (1, 2, 3):                                # the words one, two and three
            rows = np.where((y == word) & (spk == s))[0]      # this speaker's recordings of the word
            pools.append(rng.permutation(rows))               # in a random order
        n_triples = min(len(pools[0]), len(pools[1]), len(pools[2]))
        for j in range(n_triples):
            parts = [X[pools[0][j]], X[pools[1][j]], X[pools[2][j]]]   # one recording of each word
            for c in range(6):                                # join them in all six orders
                order = orders[c]
                X_out.append(np.concatenate([parts[order[0]], parts[order[1]], parts[order[2]]]))
                y_out.append(c)                               # the class is the order
                groups.append(group)                          # the six chains of a triple form one group
                speakers.append(s)
            group += 1
    return np.array(X_out), np.array(y_out, np.int64), np.array(groups), np.array(speakers)


def make_reversal(X, spk, rows):
    """The chosen recordings as recorded (class 0) and with their windows reversed (class 1)."""
    X_rev = np.concatenate([X[rows], X[rows][:, ::-1]])       # [:, ::-1] reverses the order of the windows
    y_rev = np.concatenate([np.zeros(len(rows)), np.ones(len(rows))]).astype(np.int64)
    groups = np.concatenate([np.arange(len(rows)), np.arange(len(rows))])   # a recording and its reverse
    return X_rev, y_rev, groups, np.concatenate([spk[rows], spk[rows]])


def keep_groups(X, y, groups, speakers, n):
    """A quick test keeps only n whole groups (all six chains of a triple, or both directions)."""
    unique = np.unique(groups)
    chosen = unique[np.linspace(0, len(unique) - 1, min(n, len(unique))).astype(np.int64)]
    keep = np.isin(groups, chosen)
    return X[keep], y[keep], groups[keep], speakers[keep]


if RUN_PART_1:
    tasks = {}
    Xa, ya, ga, sa = make_chains(X25_train, y_train, spk_train, 1101)     # training chains
    Xb, yb, gb, sb = make_chains(X25_test, y_test, spk_test, 1202)        # test chains
    tasks['chain'] = {'train': (Xa, ya, ga, sa), 'test': (Xb, yb, gb, sb), 'classes': 6}
    rng = np.random.default_rng(1303)
    rows = []
    for k in range(20):                                       # up to 150 training recordings of every word
        rows_k = np.where(y_train == k)[0]
        rows.append(rng.choice(rows_k, min(150, len(rows_k)), replace=False))
    rows = np.concatenate(rows)
    tasks['reversal'] = {'train': make_reversal(X25_train, spk_train, rows),
                         'test': make_reversal(X25_test, spk_test, np.arange(len(y_test))), 'classes': 2}
    tasks['negative'] = {'train': (X25_train, y_train, np.arange(len(y_train)), spk_train),
                         'test': (X25_test, y_test, np.arange(len(y_test)), spk_test), 'classes': 20}
    for name in tasks:
        if QUICK:                                             # a quick test: a few whole groups only
            Xa, ya, ga, sa = tasks[name]['train']
            tasks[name]['train'] = keep_groups(Xa, ya, ga, sa, 64)
            Xb, yb, gb, sb = tasks[name]['test']
            tasks[name]['test'] = keep_groups(Xb, yb, gb, sb, 48)
        tasks[name]['stats'] = scaling_stats(tasks[name]['train'][0])        # scaling numbers of this task
        tasks[name]['X_fit'] = to_gpu(tasks[name]['train'][0], np.arange(len(tasks[name]['train'][1])))   # on the GPU
        tasks[name]['X_test'] = to_gpu(tasks[name]['test'][0], np.arange(len(tasks[name]['test'][1])))
        print(f'{name}: {len(tasks[name]["train"][1])} training and {len(tasks[name]["test"][1])} test samples '
              f'of {tasks[name]["train"][0].shape[1]} windows, {tasks[name]["classes"]} classes')

**Part 1, training.** For every task, the two order-blind models are trained once per seed, and every
network is trained in order and on shuffled windows, five seeds each. Before training, the cell checks
that the order-blind models give one answer for every member of a test group, and times every model.

In [ ]:
if RUN_PART_1:
    folder = RESULTS_DIR + 'shd_controls/'
    meta = {'experiment': 'shd_controls', 'label': 'SHD control tasks, 25 windows per recording', 'dataset': 'SHD',
            'windows': 25, 'channels': C, 'seeds': SEEDS_USED, 'beta': BETA_25, 'designs': CONTROL_DESIGNS,
            'networks': CONTROL_NETWORKS, 'quick': QUICK, 'tasks': {},
            'environment': {'torch': torch.__version__, 'device': gpu_name()}}   # what the runs were computed on
    for name in tasks:                                        # what the analysis needs to know about each task
        ya, yb, gb, sb = tasks[name]['train'][1], tasks[name]['test'][1], tasks[name]['test'][2], tasks[name]['test'][3]
        meta['tasks'][name] = {'classes': tasks[name]['classes'], 'chance': 1 / tasks[name]['classes'],
                               'windows': int(tasks[name]['X_test'].shape[1]), 'n_train': len(ya), 'n_test': len(yb),
                               'test_labels': yb.tolist(), 'test_groups': gb.tolist(), 'test_speakers': sb.tolist()}
    save_json(folder + 'meta.json', meta)

    jobs = []                                                 # every run: (task, model, q, seed)
    for name in tasks:
        for model_name in CONTROL_DESIGNS:
            for seed in SEEDS_USED:
                jobs.append((name, model_name, 0, seed))
    for name in tasks:
        for model_name in CONTROL_NETWORKS:
            for q in (0, 1):
                for seed in SEEDS_USED:
                    jobs.append((name, model_name, q, seed))
    to_do = []                                                # the runs not saved yet
    for job in jobs:
        if not os.path.exists(folder + f'runs/{job[0]}__{job[1]}__q{job[2]}__s{job[3]}.json'):
            to_do.append(job)
    print(f'part 1: {len(jobs)} runs, {len(to_do)} still to do')

    minutes = {}                                              # estimated minutes per run, per (task, model)
    for name in tasks:
        ya, yb, gb = tasks[name]['train'][1], tasks[name]['test'][1], tasks[name]['test'][2]
        K_task, stats = tasks[name]['classes'], tasks[name]['stats']
        X_fit, X_test = tasks[name]['X_fit'], tasks[name]['X_test']
        if RUN_CHECKS:                                        # order-blind: one answer for a whole test group
            group = np.where(gb == gb[0])[0]                  # the samples of the first test group
            x = get_batch(X_test, group)
            x_shuffled = get_batch(X_test, group, orders_to_gpu(window_orders(len(yb), X_test.shape[1], 5)))
            for model_name in CONTROL_DESIGNS:
                torch.manual_seed(0)
                model = make_model(model_name, K_task, BETA_25, stats).to(DEVICE).eval()
                with torch.no_grad():
                    out, out_shuffled = model(x), model(x_shuffled)
                assert torch.allclose(out, out[:1].expand_as(out), atol=1e-4), model_name + ' must give one answer per group'
                assert torch.allclose(out, out_shuffled, atol=1e-4), model_name + ' must ignore the window order'
            print(f'   {name}: the order-blind models give one answer for every member of a test group')
        task_minutes, runs = {}, {}
        for model_name in CONTROL_DESIGNS + CONTROL_NETWORKS:
            n = 0
            for job in to_do:
                if job[0] == name and job[1] == model_name:
                    n += 1
            if n > 0:
                task_minutes[model_name] = minutes_per_run(model_name, K_task, BETA_25, stats, X_fit, ya)
                runs[model_name] = n
                minutes[(name, model_name)] = task_minutes[model_name]
        if len(runs) > 0:
            print(f'\n{name}: estimated time')
            print_estimate(task_minutes, runs)

    total = 0.0
    for job in to_do:
        total += minutes[(job[0], job[1])]
    bar = tqdm(total=total, desc='part 1', bar_format=BAR_FORMAT)   # the progress bar of part 1
    for name, model_name, q, seed in to_do:
        start = time.time()
        ya, yb = tasks[name]['train'][1], tasks[name]['test'][1]
        K_task, stats = tasks[name]['classes'], tasks[name]['stats']
        X_fit, X_test = tasks[name]['X_fit'], tasks[name]['X_test']
        T = X_fit.shape[1]
        if name == 'negative':                                # the negative control: windows shuffled first
            X_fit = reorder(X_fit, window_orders(len(ya), T, NEGATIVE_SEED_FIT + 10 * seed))
            X_test = reorder(X_test, window_orders(len(yb), T, NEGATIVE_SEED_TEST + 10 * seed))
        order_fit, order_test = None, None
        if q == 1:                                            # the shuffled arm
            order_fit = orders_to_gpu(window_orders(len(ya), T, SHUFFLE_SEED_FIT + 10 * seed))
            order_test = orders_to_gpu(window_orders(len(yb), T, SHUFFLE_SEED_TEST + 10 * seed))
        torch.manual_seed(seed)                               # the seed fixes the starting weights ...
        np.random.seed(seed)                                  # ... and the order of the recordings in training
        model = make_model(model_name, K_task, BETA_25, stats).to(DEVICE)
        train_model(model, X_fit, ya, order_fit, recipe_of(model_name), label=f'{name} {model_name} q{q} s{seed}')
        train_acc = test_model(model, X_fit, ya, order_fit, EVAL_SEED + seed)[0]
        test_acc, answers, spikes, _ = test_model(model, X_test, yb, order_test, EVAL_SEED + seed)
        record = {'experiment': 'shd_controls', 'task': name, 'model': model_name, 'q': q, 'seed': seed,
                  'train_acc': train_acc, 'test_acc': test_acc, 'preds': answers.tolist(),
                  'spikes_per_recording': spikes, 'seconds': time.time() - start, 'device': gpu_name()}
        save_json(folder + f'runs/{name}__{model_name}__q{q}__s{seed}.json', record)
        bar.update(min(minutes[(name, model_name)], bar.total - bar.n))
        tqdm.write(f'{name:9s} {model_name:9s} q{q} s{seed}: train {train_acc:.3f}  test {test_acc:.4f}')
        del model
    bar.n = bar.total                                         # the estimate is approximate: show the bar full
    bar.close()
    for name in tasks:                                        # free the GPU memory
        del tasks[name]['X_fit'], tasks[name]['X_test']

## Step 14. Running parts 2 and 3

`prepare_data` holds out 10% of the training recordings, computes the scaling numbers and the input
statistics, and puts the tables on the GPU.

`run_one` trains one model once and saves its result. With `q = 0` the windows are in order; with
`q = 1` every recording's windows are in its own random order, for the fitting, held-out and test
recordings alike. After training it measures the accuracy on the fitting, held-out and test
recordings, the hidden spikes and the energy estimate. In part 2, a network trained in order is also
tested on shuffled windows without retraining (25%, 50%, 75% and 100% of the windows moved).

`run_experiment` saves a description of the experiment, runs the self-checks, prints the time
estimate, and then calls `run_one` for every run still to do, with a progress bar.

In [ ]:
def prepare_data(X_train, y_train, spk_train, X_test, y_test, spk_test, T):
    """The hold-out split, the scaling numbers, the input statistics, and the tables on the GPU."""
    fit_rows, val_rows = holdout(y_train, HOLDOUT, VAL_SEED)
    test_rows = np.arange(len(y_test))
    if QUICK:                                                 # a quick test: a few recordings only
        fit_rows, val_rows, test_rows = thin(fit_rows, 384), thin(val_rows, 128), thin(test_rows, 256)
    data = {'T': T}
    data['stats'] = scaling_stats(X_train[fit_rows])          # from the fitting recordings only
    data['input'] = input_statistics(X_test[test_rows])       # the test recordings' size, for the energy
    data['X_fit'], data['y_fit'] = to_gpu(X_train, fit_rows), y_train[fit_rows]
    data['X_val'], data['y_val'] = to_gpu(X_train, val_rows), y_train[val_rows]
    data['X_test'], data['y_test'] = to_gpu(X_test, test_rows), y_test[test_rows]
    data['test_speakers'] = spk_test[test_rows]               # who spoke each test recording
    data['train_speakers'] = sorted(set(spk_train.tolist()))  # the speakers heard in training
    print(f'fitting {len(fit_rows)}, held out {len(val_rows)}, test {len(test_rows)} recordings of {T} windows')
    return data


def run_one(experiment, model_name, q, seed, data, beta, folder, eval_only_shares):
    """Train one model once (q = 0: windows in order; q = 1: shuffled), test it, and save the result."""
    start = time.time()
    T = data['T']
    order_fit, order_val, order_test = None, None, None
    if q == 1:                                                # the shuffled arm: one random order per recording
        order_fit = orders_to_gpu(window_orders(len(data['y_fit']), T, SHUFFLE_SEED_FIT + 10 * seed))
        order_val = orders_to_gpu(window_orders(len(data['y_val']), T, SHUFFLE_SEED_VAL + 10 * seed))
        order_test = orders_to_gpu(window_orders(len(data['y_test']), T, SHUFFLE_SEED_TEST + 10 * seed))
    recipe = recipe_of(model_name)
    torch.manual_seed(seed)                                   # the seed fixes the starting weights ...
    np.random.seed(seed)                                      # ... and the order of the recordings in training
    model = make_model(model_name, K, beta, data['stats']).to(DEVICE)
    curve, best_epoch = train_model(model, data['X_fit'], data['y_fit'], order_fit, recipe,
                                    data['X_val'], data['y_val'], order_val, EVAL_SEED + seed,
                                    label=f'{experiment} {model_name} q{q} s{seed}')
    eval_seed = EVAL_SEED + seed
    train_acc = test_model(model, data['X_fit'], data['y_fit'], order_fit, eval_seed)[0]
    val_acc = test_model(model, data['X_val'], data['y_val'], order_val, eval_seed)[0]
    test_acc, answers, spikes, layer_spikes = test_model(model, data['X_test'], data['y_test'], order_test, eval_seed)
    record = {'experiment': experiment, 'model': model_name, 'q': q, 'seed': seed, 'recipe': recipe,
              'train_acc': train_acc, 'val_acc': val_acc, 'test_acc': test_acc, 'preds': answers.tolist(),
              'spikes_per_recording': spikes, 'layer_spikes_per_recording': layer_spikes,
              'energy_pJ': model.energy(data['input'], spikes, layer_spikes), 'best_epoch': best_epoch,
              'val_curve': curve if curve else None, 'weights': n_weights(model), 'device': gpu_name()}
    if q == 0 and len(eval_only_shares) > 0:                  # tested on shuffled windows without retraining
        record['eval_only'] = {}
        for i in range(len(eval_only_shares)):
            share = eval_only_shares[i]
            order = orders_to_gpu(window_orders(len(data['y_test']), T, EVAL_ONLY_SEED + 10 * seed + i, share))
            record['eval_only'][str(share)] = test_model(model, data['X_test'], data['y_test'], order, eval_seed)[0]
    if model_name == 'radlif':                                # RadLIF's learned time constants
        record['tau_mem_windows_quartiles'] = model.tau_quartiles()
    record['seconds'] = time.time() - start
    save_json(folder + f'runs/{model_name}__q{q}__s{seed}.json', record)
    return record


def run_experiment(experiment, label, data, designs, networks, beta, window_mode, window_ms, eval_only_shares):
    """Every run of one experiment: the designs once per seed; the networks in order and shuffled."""
    folder = RESULTS_DIR + experiment + '/'
    meta = {'experiment': experiment, 'label': label, 'dataset': 'SHD', 'window_mode': window_mode, 'windows': data['T'],
            'window_ms': window_ms, 'channels': C, 'classes': K, 'chance': 1 / K,
            'competence_bar': 1 / K + 0.5 * (1 - 1 / K),  # training accuracy halfway from chance to perfect
            'seeds': SEEDS_USED, 'holdout_share': HOLDOUT, 'n_fit': len(data['y_fit']), 'n_val': len(data['y_val']),
            'n_test': len(data['y_test']), 'beta': beta, 'input_per_test_recording': data['input'],
            'test_labels': data['y_test'].tolist(), 'test_speakers': data['test_speakers'].tolist(),
            'train_speakers': data['train_speakers'], 'designs': designs, 'networks': networks,
            'energy_costs_pJ': COSTS, 'quick': QUICK, 'models': {},
            'environment': {'torch': torch.__version__, 'device': gpu_name()}}   # what the runs were computed on
    for name in designs + networks:                           # every model's kind and size
        torch.manual_seed(0)
        model = make_model(name, K, beta, data['stats'])
        if name in LARGE_DESIGNS or name == 'radlif':
            scale = 'RadLIF'
        else:
            scale = '128 units'
        if name in designs:
            kind = 'design'
        else:
            kind = 'network'
        meta['models'][name] = {'kind': kind, 'family': model.family, 'weights': n_weights(model),
                                'recipe': recipe_of(name), 'scale': scale}
    save_json(folder + 'meta.json', meta)

    if RUN_CHECKS:
        print(f'{experiment}: self-checks')
        x = get_batch(data['X_test'], np.arange(32))          # 32 test recordings in order ...
        shuffled_orders = orders_to_gpu(window_orders(len(data['y_test']), data['T'], 5))
        x_shuffled = get_batch(data['X_test'], np.arange(32), shuffled_orders)   # ... and shuffled
        check_order_blind(designs + networks, K, beta, data['stats'], x[:4], x_shuffled[:4])
        if 'lif_std' in networks or 'rlif' in networks:
            check_against_snntorch(K, beta, data['stats'], x, data['y_test'][:32])
        if 'radlif' in networks:
            check_radlif(K, x)

    jobs = []                                                 # every run: (model, q, seed)
    for name in designs:
        for seed in SEEDS_USED:
            jobs.append((name, 0, seed))
    for q in (0, 1):
        for name in networks:
            for seed in SEEDS_USED:
                jobs.append((name, q, seed))
    to_do = []                                                # the runs not saved yet
    for job in jobs:
        if not os.path.exists(folder + f'runs/{job[0]}__q{job[1]}__s{job[2]}.json'):
            to_do.append(job)
    print(f'{experiment}: {len(jobs)} runs, {len(to_do)} still to do')
    if len(to_do) == 0:
        return

    minutes, runs = {}, {}                                    # the time estimate
    for name in designs + networks:
        n = 0
        for job in to_do:
            if job[0] == name:
                n += 1
        if n > 0:
            minutes[name] = minutes_per_run(name, K, beta, data['stats'], data['X_fit'], data['y_fit'])
            runs[name] = n
    print(f'\n{experiment}: estimated time of the runs still to do')
    print_estimate(minutes, runs)

    total = 0.0
    for job in to_do:
        total += minutes[job[0]]
    bar = tqdm(total=total, desc=experiment, bar_format=BAR_FORMAT)   # the progress bar of the experiment
    for name, q, seed in to_do:
        if name in networks:
            shares = eval_only_shares
        else:
            shares = []
        r = run_one(experiment, name, q, seed, data, beta, folder, shares)
        bar.update(min(minutes[name], bar.total - bar.n))
        e = r['energy_pJ']
        tqdm.write(f'{name:22s} q{q} s{seed}: train {r["train_acc"]:.3f}  val {r["val_acc"]:.4f}  test {r["test_acc"]:.4f}'
                   f'  | energy {e["worst"] / 1e6:.3g} / {e["within"] / 1e6:.3g} / {e["across"] / 1e6:.3g} uJ  | {r["seconds"]:.0f} s')
        if r['train_acc'] < meta['competence_bar'] and not QUICK:
            tqdm.write(f'   note: training accuracy below the competence bar ({meta["competence_bar"]:.4f}): reported, not interpreted')
    bar.n = bar.total                                         # the estimate is approximate: show the bar full
    bar.close()

## Part 2. SHD, 25 windows per recording

Every recording is cut into 25 windows spanning it (about 28 ms each, on average). 9 order-blind
designs; lif_std, rlif, rnn, the LSTM at four input-dropout levels, and RadLIF. Five seeds.

In [ ]:
if RUN_PART_2:
    data = prepare_data(X25_train, y_train, spk_train, X25_test, y_test, spk_test, 25)
    window_ms = 1000 * float(np.concatenate([dur_train, dur_test]).mean()) / 25   # the average window length
    run_experiment('shd_25', 'SHD, 25 windows spanning each recording', data, DESIGNS, NETWORKS_25, BETA_25,
                   'relative', window_ms, [0.25, 0.5, 0.75, 1.0])
    del data                                                  # free the GPU memory for part 3
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

## Part 3. SHD, 100 windows of 14 ms

Every recording is cut into 100 windows of 14 ms from time 0 (1.4 s in all). The same 9 order-blind
designs; RadLIF, rlif and the LSTM at four input-dropout levels. Five seeds. The spiking neurons keep
the same memory in milliseconds as at 25 windows.

In [ ]:
if RUN_PART_3:
    X14_train, y_train, spk_train, dur_train = count_spikes(SHD_TRAIN, 'fixed', 100, 0.014)
    X14_test, y_test, spk_test, dur_test = count_spikes(SHD_TEST, 'fixed', 100, 0.014)
    data = prepare_data(X14_train, y_train, spk_train, X14_test, y_test, spk_test, 100)
    run_experiment('shd_14ms', 'SHD, 100 windows of 14 ms', data, DESIGNS, NETWORKS_14, BETA_14, 'fixed', 14.0, [])
    del data

## Step 15. Where the results are

A full run's results are on your Google Drive. A quick test's results are on this Colab machine, which
is deleted when the session ends, so they are packed into one `.zip` file and offered as a download.
When `03_train_SSC.ipynb` has finished too, `04_analyze_results.ipynb` makes the tables and figures
from your own runs (with `RESULTS = 'mine'` in its settings).

In [ ]:
if USE_DRIVE:
    print('The results are on your Google Drive, in', RESULTS_DIR)
else:
    archive = shutil.make_archive(RESULTS_DIR.rstrip('/'), 'zip', RESULTS_DIR)   # one .zip file of the folder
    print('The results are in', archive)
    try:
        from google.colab import files        # Colab's tool for saving a file to your computer
        files.download(archive)               # your browser saves the .zip file
    except ImportError:                       # not on Colab: the file stays where it is
        pass